In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import Window, functions as F


In [2]:
S3_BUCKET = "datalake-teste2"
S3_BRONZE = f"s3a://{S3_BUCKET}/bronze"
S3_SILVER = f"s3a://{S3_BUCKET}/silver"
S3_GOLD = f"s3a://{S3_BUCKET}/gold"

SPARK_MASTER = "local[*]"
SPARK_APP_NAME = "medallion-pipeline"

print(f"Bronze:  {S3_BRONZE}")
print(f"Silver:  {S3_SILVER}")
print(f"Gold:    {S3_GOLD}")

Bronze:  s3a://datalake-teste2/bronze
Silver:  s3a://datalake-teste2/silver
Gold:    s3a://datalake-teste2/gold


In [3]:
spark = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .master(SPARK_MASTER) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
    .getOrCreate()

26/09/03 00:46:36 WARN Utils: Your hostname, hugo resolves to a loopback address: 127.0.1.1; using 10.159.141.203 instead (on interface wlp2s0)
26/09/03 00:46:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hugo/Desktop/Project/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hugo/.ivy2/cache
The jars for the packages stored in: /home/hugo/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6955d3cb-ad54-4179-821b-6e6e52814c7d;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.261 in central
:: resolution report :: resolve 426ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.261 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 by [com.amazonaws#aws-java-sdk-bundle;1.12.261] in [default]
	------------------------------------------------------------------

In [4]:
df = spark.read.parquet(f"{S3_SILVER}/eventos_unificados")

print(f"\n📥 {df.count()} eventos lidos de eventos_unificados")

26/09/03 00:46:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties



📥 36 eventos lidos de eventos_unificados


In [5]:
# Colunas que identificam o evento, presentes em todas as fontes.
CHAVES = ["transaction_datetime", "transaction_date", "purchase_id", "origem_evento"]


COLUNAS_POR_FONTE = {
    "purchase": ["buyer_id", "prod_item_id", "order_date", "release_date", "producer_id"],
    "product_item": ["product_id", "item_quantity", "purchase_value"],
    "purchase_extra_info": ["subsidiary"],
}


In [6]:
df = spark.read.parquet(f"{S3_SILVER}/purchase_historico").select(
        "transaction_date",
        "purchase_id",
        "release_date",
        "purchase_value",
        F.coalesce("subsidiary", F.lit('nao_informada')).alias("subsidiary"),
    )

In [7]:
df.show()

+----------------+-----------+------------+--------------+-------------+
|transaction_date|purchase_id|release_date|purchase_value|   subsidiary|
+----------------+-----------+------------+--------------+-------------+
|      2023-01-23|         55|  2023-01-20|         50.00|     nacional|
|      2023-03-12|         69|  2023-02-28|       2000.00|internacional|
|      2023-02-28|         69|  2023-02-28|       2000.00|     nacional|
|      2023-09-01|         70|  2023-03-02|        330.00|internacional|
|      2023-06-25|         76|  2023-06-25|        150.00|internacional|
|      2023-04-04|         72|  2023-04-04|       1200.00|     nacional|
|      2023-03-02|         70|  2023-03-02|        300.00|     nacional|
|      2023-05-22|         74|  2023-05-22|        480.00|internacional|
|      2023-08-03|         70|  2023-03-02|        330.00|     nacional|
|      2023-07-12|         55|  2023-01-20|         55.00|     nacional|
|      2023-06-08|         75|  2023-06-08|        

In [8]:
# A contribuição de uma versão para o GMV é a tripla
# (release_date, subsidiary, purchase_value). Para saber o que mudou,
# comparamos cada versão com a anterior daquela compra — é por isso que o
# SCD2 precisava vir antes: sem versões ordenadas não há "valor anterior".
versoes = Window.partitionBy("purchase_id").orderBy("transaction_date")

df = (df
      .withColumn("release_date_ant", F.lag("release_date").over(versoes))
      .withColumn("subsidiary_ant", F.lag("subsidiary").over(versoes))
      .withColumn("purchase_value_ant", F.lag("purchase_value").over(versoes))
)

# Lançamento: a versão atual entra em cena, se o pagamento foi confirmado.
# release_date NULL = pagamento não efetuado (ou cancelado) -> não é GMV.
lancamento = df.filter(F.col("release_date").isNotNull()).select(
        "transaction_date",
        "release_date",
        "subsidiary",
        F.col("purchase_value").alias("gmv_delta"),
)